kernel : Python (Pixi)

In [1]:
import anndata
import numpy as np
import pandas as pd
import os
import scanpy as sc
from anndata import AnnData
from scipy import sparse
from scipy.sparse import csr_matrix
from pipeline.utils.env import find_env_dir
from pipeline.utils import plot

anndata.settings.allow_write_nullable_strings = True

series_name = "Zheng"
clustered_data_location = find_env_dir("CLUSTERED_DATA")

zheng_adata = sc.read_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [ ]:
cell_marker_genes = {
    "OPC": ["Pdgfra", "Megf11", "Vcan", "Has2", "Col9a1", "Myt1", ],
    "COP": ["Gpr17", "Tcf7l2", ],
    "Oligodendrocyte": ["Mbp", "Mog", "Mag", "Cnp", "Mobp", "Fa2h",
                        "Ppp1r14a", "Ermn", "Gpr37", "Myrf", ],
    "Neuron": ["Stmn2", "Rbfox3", "Eno2", "Syt1", "Cck", "Grin1", "Tmem130", ],
    "Microglia": ["Tmem119", "Siglech", "Olfml3", "Gpr34", ],
    "Lymphocyte": ["Cd3d", "Cd3e", "Cd3g", "Trac", "Cd2", "Cd5", "Lck", "Cd79a", "Cd79b", "Ms4a1", "Cd19", ],
    "BAM": ["Mrc1", "Cd163", "Lyve1", "Pf4", "Ms4a7", "Stab1", "Cbr2", ], 
    "Macrophage": [ "Ly6c2", "Lgals3", "S100a11", "Ccl5", "Ccr2", "Tgfbi"],
    "Endothelial": ["Cldn5", "Cd34", "Flt1", "Vwf", "Cdh5", "Clec14a", "Erg", "Itm2a", ],
    "Pericyte": ["Rgs5", "Cd248", "Kcnj8", ],
    "Astrocyte": ["Aqp4", "Aldh1l1", "Atp1b2", "Aldh1a1", "Agt", "Atp13a4", "Bmpr1b", "Rfx4", "Slc4a4", ],
    "Ependymal": ["Foxj1", "Dynlrb2", "Dnah11", "Pifo", ],
    "VLMC": ["Col1a2", "Col3a1", ],
}

plot.plot_dotplot(zheng_adata, series_name, cell_marker_genes, group = "leiden")

In [5]:
leiden_to_celltype = {
    "0": "Astrocyte",
    "1": "Oligodendrocyte",
    "2": "Oligodendrocyte",
    "3": "Oligodendrocyte",
    "4": "Oligodendrocyte",
    "5": "Oligodendrocyte",
    "6": "Oligodendrocyte",
    "7": "Oligodendrocyte",
    "8": "Oligodendrocyte",
    "9": "Oligodendrocyte",
    "10": "Oligodendrocyte",
    "11": "Oligodendrocyte",
    "12": "Oligodendrocyte",
    "13": "Oligodendrocyte",
    "14": "Oligodendrocyte",
    "15": "Oligodendrocyte",
    "16": "COP",
    "17": "OPC",
    "18": "Oligodendrocyte",
    "19": "Oligodendrocyte",
    "20": "OPC",
    "21": "Oligodendrocyte",
    "22": "Oligodendrocyte",
    "23": "Microglia",
    "24": "Oligodendrocyte",
    "25": "Oligodendrocyte",
    "26": "Oligodendrocyte",
    "27": "VLMC",
    "28": "Oligodendrocyte",
    "29": "Oligodendrocyte",
    "30": "Oligodendrocyte",
    "31": "Oligodendrocyte",
    "32": "Oligodendrocyte",
    "33": "Oligodendrocyte",
    "34": "Astrocyte",
    "35": "Oligodendrocyte",
    "36": "Oligodendrocyte",
    "37": "Oligodendrocyte",
    "38": "Oligodendrocyte",
    "39": "Oligodendrocyte",
    "40": "Oligodendrocyte",
    "41": "Oligodendrocyte",
    "42": "Oligodendrocyte",
    "43": "Oligodendrocyte",
    "44": "Oligodendrocyte",
    "45": "Doublet",
    "46": "Oligodendrocyte",
    "47": "Macrophage",
    "48": "Oligodendrocyte",
    "49": "Ependymal",
    "50": "Astrocyte",
    "51": "Neuron",
    "52": "Oligodendrocyte",
    "53": "Lymphocyte",
    "54": "Oligodendrocyte",
    "55": "Endothelial+Pericyte",
    "56": "Astrocyte",
    "57": "BAM",
    "58": "Oligodendrocyte",
}

leiden_str = zheng_adata.obs["leiden"].astype(str)
mapped = leiden_str.map(leiden_to_celltype)

missing = sorted(leiden_str[mapped.isna()].unique())
if missing:
    raise ValueError(
        f"Missing leiden value in leiden_to_celltype : {missing}\n"
    )
zheng_adata.obs["cell"] = mapped

In [6]:
zheng_adata.write_h5ad(os.path.join(clustered_data_location, series_name + "_clustered.h5ad"))

In [7]:
del zheng_adata.obsp

h5ad_matrix_location = find_env_dir("H5AD_MATRIX")
opc = zheng_adata[
    (zheng_adata.obs["cell"] == "OPC") |
    (zheng_adata.obs["cell"] == "COP")
]
opc.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_opc.h5ad"))

oligodendrocyte = zheng_adata[
    zheng_adata.obs["cell"] == "Oligodendrocyte"
]
oligodendrocyte.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligodendrocyte.h5ad"))

oligo = zheng_adata[
    (zheng_adata.obs["cell"] == "OPC") |
    (zheng_adata.obs["cell"] == "COP") | 
    (zheng_adata.obs["cell"] == "Oligodendrocyte")
]
oligo.write_h5ad(os.path.join(h5ad_matrix_location, series_name + "_oligo.h5ad"))